# Can we predict whether a student passes the closing quiz?
Target: `passed_quiz`. Follows `checklist.md`.

**Design decisions (made before any modelling)**
- `data.csv` has no time column and each student appears in several rows (~7 on average), so the test set holds **whole students** (checklist 1), and CV inside training uses **whole-student folds** (`GroupKFold`).
- Identifier `student_key` is dropped as a feature (used only for grouping).
- Features: `enrol_year`, `school_code` (categorical), `prior_score`, `session_minutes`, `hints_used`. Session length and hints are recorded during the session, i.e. before the closing quiz result, so they are available when the quiz outcome is being predicted. If the model must predict *before the session starts*, drop those two columns; this is a caveat on the headline.
- Preprocessing (scaling, one-hot) lives inside a `Pipeline`, so it is fit within each CV fold.

In [ ]:
import numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, accuracy_score, brier_score_loss
df = pd.read_csv('data.csv')
print(df.shape, 'students:', df.student_key.nunique(), 'pass rate: %.3f' % df.passed_quiz.mean())
df.describe().T

## 1. Set aside the test set (whole students, 20%)

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
tr_idx, te_idx = next(gss.split(df, groups=df.student_key))
train, test = df.iloc[tr_idx].reset_index(drop=True), df.iloc[te_idx].reset_index(drop=True)
assert set(train.student_key).isdisjoint(test.student_key)
FEATS = ['enrol_year','school_code','prior_score','session_minutes','hints_used']
X, y, g = train[FEATS], train.passed_quiz, train.student_key
print(len(train), len(test), 'train pass rate %.3f' % y.mean())
# The test set is not touched again until step 5.

## 2. Candidate models (named before any is scored)
1. `baseline`: predicts the training prevalence (trivial).
2. `logreg`: logistic regression on scaled numerics + one-hot school, C in {0.1, 1}.
3. `logreg_nosch`: same without `school_code` and `enrol_year` (tests whether those add anything).
4. `hgb`: histogram gradient boosting, max_depth in {2, 4}, small learning rate.

Selection metric: mean out-of-fold ROC AUC (threshold-free). The decision threshold is then chosen on out-of-fold predictions of the winner by maximising balanced accuracy.

In [ ]:
def pre(cols):
    nums = [x for x in cols if x!='school_code']
    t = [('n', StandardScaler(), nums)]
    if 'school_code' in cols:
        t.append(('c', OneHotEncoder(handle_unknown='ignore'), ['school_code']))
    return ColumnTransformer(t)
cands = {'baseline': (DummyClassifier(strategy='prior'), FEATS)}
f2 = ['prior_score','session_minutes','hints_used']
for C_ in (0.1, 1):
    cands[f'logreg C={C_}'] = (Pipeline([('p', pre(FEATS)), ('m', LogisticRegression(C=C_, max_iter=2000))]), FEATS)
    cands[f'logreg_nosch C={C_}'] = (Pipeline([('p', pre(f2)), ('m', LogisticRegression(C=C_, max_iter=2000))]), f2)
for d in (2, 4):
    cands[f'hgb depth={d}'] = (Pipeline([('p', 'passthrough'),
        ('m', HistGradientBoostingClassifier(max_depth=d, learning_rate=0.05, max_iter=200, random_state=0))]), FEATS)
list(cands)

## 3. Grouped cross-validation on training rows only

In [ ]:
folds = list(GroupKFold(n_splits=5).split(X, y, groups=g))
oof, res = {}, []
for name,(est,cols) in cands.items():
    p = np.zeros(len(train)); aucs=[]
    for a,b in folds:
        m = clone(est).fit(X.iloc[a][cols], y.iloc[a])
        p[b] = m.predict_proba(X.iloc[b][cols])[:,1]
        aucs.append(roc_auc_score(y.iloc[b], p[b]))
    oof[name]=p; res.append((name, np.mean(aucs), np.std(aucs)))
cvres = pd.DataFrame(res, columns=['model','cv_auc_mean','cv_auc_sd']).sort_values('cv_auc_mean', ascending=False)
cvres

## 4. Choose model and threshold on CV only

In [ ]:
best = cvres.iloc[0].model
ths = np.linspace(0.2, 0.7, 51)
bal = [balanced_accuracy_score(y, oof[best] >= t) for t in ths]
THR = float(ths[int(np.argmax(bal))])
print('chosen model:', best, '| threshold: %.2f' % THR, '| OOF balanced acc: %.3f' % max(bal))
print('baseline CV AUC: %.3f' % cvres.set_index('model').loc['baseline','cv_auc_mean'])

**Recorded choice (fixed before scoring the test set):** the model and threshold printed above. Nothing below changes them.

## 5. Refit on all training rows; score once on the test set (headline)

In [ ]:
est, cols = cands[best]
final = clone(est).fit(X[cols], y)
pt = final.predict_proba(test[cols])[:,1]
yt = test.passed_quiz
print('TEST  AUC %.3f | balanced acc %.3f | accuracy %.3f | Brier %.3f' % (
    roc_auc_score(yt, pt), balanced_accuracy_score(yt, pt>=THR), accuracy_score(yt, pt>=THR), brier_score_loss(yt, pt)))
print('majority-class accuracy on test: %.3f' % max(yt.mean(), 1-yt.mean()))
# uncertainty: bootstrap whole students in the test set (descriptive only, no refitting)
rng = np.random.default_rng(0); ids = test.student_key.unique(); grp = {k:v.index.values for k,v in test.groupby('student_key')}
bs=[]
for _ in range(500):
    idx = np.concatenate([grp[k] for k in rng.choice(ids, len(ids))])
    bs.append(roc_auc_score(yt.iloc[idx], pt[idx]))
print('test AUC 95%% CI (student bootstrap): %.3f - %.3f' % tuple(np.percentile(bs,[2.5,97.5])))

## Conclusion (from the run of this notebook)
Weakly predictable. Chosen on CV: logistic regression on `prior_score`, `session_minutes`, `hints_used` (C=1), threshold 0.41; CV AUC 0.604 vs 0.500 baseline. **Headline, test set scored once: AUC 0.592 (95% student-bootstrap CI 0.570-0.616), balanced accuracy 0.569, accuracy 0.565, which is below the 0.582 majority-class accuracy.** Signal comes mainly from `session_minutes` (longer helps) and `hints_used` (more hints, lower pass rate). Adding `school_code`/`enrol_year` hurt in CV, and boosting did not beat the linear model. Caveats: no time column, so this only speaks to new students, not future periods; `session_minutes`/`hints_used` are only known once the session is under way.